# Deep Learning orientado a ranking

En este notebook se desarrolla una variante del sistema de recomendación basada en Deep Learning cuyo objetivo principal es mejorar la calidad del ranking de películas recomendadas.

El modelo Deep Learning desarrollado anteriormente obtuvo buenos resultados en la predicción de valoraciones, pero presentó un rendimiento inferior al modelo de factores latentes en las métricas Top-N.

Por este motivo, se plantea una nueva estrategia de entrenamiento orientada directamente a distinguir entre películas relevantes y no relevantes para cada usuario.

## 1. Preparación del entorno

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [2]:
print("TensorFlow:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
GPU disponible: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 2. Carga de datos

Se reutilizan las particiones generadas durante el desarrollo anterior para mantener la coherencia entre los experimentos.

El nuevo modelo se entrenará utilizando exclusivamente el conjunto de entrenamiento, mientras que validación se utilizará para controlar el aprendizaje y test permanecerá reservado para la evaluación final.

In [4]:
ruta_modelos = "/content/drive/MyDrive/TFM/modelos"

train = pd.read_csv(f"{ruta_modelos}/train.csv")
validation = pd.read_csv(f"{ruta_modelos}/validation.csv")
test = pd.read_csv(f"{ruta_modelos}/test.csv")

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (80668, 6)
Validation: (10084, 6)
Test: (10084, 6)


In [5]:
ruta_datos = (
    "/content/drive/MyDrive/TFM/"
    "Recomendador_Peliculas/data/raw/ml-latest-small"
)

movies = pd.read_csv(
    f"{ruta_datos}/movies.csv"
)

print("Movies:", movies.shape)
display(movies.head())

Movies: (9742, 3)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


## 3. Transformación a un problema de ranking

El modelo anterior fue entrenado como un problema de regresión, tratando de aproximar directamente la valoración otorgada por el usuario.

Sin embargo, para generar recomendaciones Top-N resulta más importante distinguir qué películas son relevantes para un usuario y situarlas por encima de aquellas que probablemente no lo sean.

Por este motivo, el nuevo modelo se plantea como un problema de preferencia implícita, considerando como interacciones positivas aquellas valoraciones iguales o superiores a 4 puntos.

### 3.1. Selección de interacciones positivas

Se consideran positivas aquellas interacciones del conjunto de entrenamiento en las que el usuario ha otorgado una valoración igual o superior a 4.

Estas interacciones representan películas que el usuario ha valorado favorablemente y que el modelo deberá aprender a priorizar.

In [6]:
umbral_relevancia = 4.0

train_positivos = train[
    train["rating"] >= umbral_relevancia
].copy()

validation_positivos = validation[
    validation["rating"] >= umbral_relevancia
].copy()

print("Positivos train:", len(train_positivos))
print("Positivos validation:", len(validation_positivos))

print(
    "Porcentaje positivo en train:",
    round(len(train_positivos) / len(train) * 100, 2),
    "%"
)

Positivos train: 38889
Positivos validation: 4938
Porcentaje positivo en train: 48.21 %


### 3.2. Generación de ejemplos negativos

MovieLens contiene las películas que los usuarios han valorado, pero no proporciona ejemplos explícitos de películas que hayan sido descartadas por el usuario.

Para entrenar el modelo orientado a ranking se generan ejemplos negativos mediante muestreo. Por cada interacción positiva se seleccionan cuatro películas que el usuario no ha valorado.

Las interacciones positivas reciben la etiqueta 1 y las negativas la etiqueta 0. De esta forma, el modelo aprende a asignar puntuaciones superiores a las películas relevantes frente a películas para las que no existe evidencia de interés.

In [7]:
valoraciones_completas = pd.concat(
    [
        train[["userId", "movieId"]],
        validation[["userId", "movieId"]],
        test[["userId", "movieId"]]
    ],
    ignore_index=True
)

peliculas_vistas_usuario = (
    valoraciones_completas
    .groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

print(
    "Usuarios con historial:",
    len(peliculas_vistas_usuario)
)

Usuarios con historial: 610


In [8]:
peliculas_train = np.array(
    sorted(train["movieId"].unique())
)

print(
    "Películas disponibles:",
    len(peliculas_train)
)

Películas disponibles: 8983


In [9]:
rng = np.random.default_rng(42)

num_negativos = 4

ejemplos = []

for fila in train_positivos.itertuples():

    usuario = fila.userId
    pelicula_positiva = fila.movieId

    # Ejemplo positivo
    ejemplos.append(
        (usuario, pelicula_positiva, 1)
    )

    vistas = peliculas_vistas_usuario[usuario]

    negativas_disponibles = peliculas_train[
        ~np.isin(
            peliculas_train,
            list(vistas)
        )
    ]

    peliculas_negativas = rng.choice(
        negativas_disponibles,
        size=num_negativos,
        replace=False
    )

    # Ejemplos negativos
    for pelicula_negativa in peliculas_negativas:

        ejemplos.append(
            (usuario, pelicula_negativa, 0)
        )

In [10]:
train_ranking = pd.DataFrame(
    ejemplos,
    columns=[
        "userId",
        "movieId",
        "label"
    ]
)

display(train_ranking.head(10))

print(
    "Tamaño del dataset:",
    train_ranking.shape
)

print()
print(
    "Distribución de etiquetas:"
)

print(
    train_ranking["label"].value_counts()
)

,userId,movieId,label
0,326,71462,1
1,326,5879,0
2,326,1084,0
3,326,45950,0
4,326,80241,0
5,610,1127,1
6,610,52299,0
7,610,2349,0
8,610,1064,0
9,610,7076,0


Tamaño del dataset: (194445, 3)

Distribución de etiquetas:
label
0    155556
1     38889
Name: count, dtype: int64


## 4. Codificación de usuarios y películas

Las capas de embeddings necesitan recibir índices enteros consecutivos. Por este motivo, los identificadores originales de usuarios y películas se transforman en índices internos.

El mapeo se construye exclusivamente a partir del conjunto de entrenamiento, evitando incorporar información procedente de validación o test al proceso de aprendizaje.

In [11]:
usuarios = train["userId"].unique()
peliculas = train["movieId"].unique()

usuario_a_indice = {
    usuario: indice
    for indice, usuario in enumerate(usuarios)
}

pelicula_a_indice = {
    pelicula: indice
    for indice, pelicula in enumerate(peliculas)
}

num_usuarios = len(usuario_a_indice)
num_peliculas = len(pelicula_a_indice)

print("Usuarios:", num_usuarios)
print("Películas:", num_peliculas)

Usuarios: 610
Películas: 8983


In [12]:
train_ranking["usuario_idx"] = (
    train_ranking["userId"]
    .map(usuario_a_indice)
)

train_ranking["pelicula_idx"] = (
    train_ranking["movieId"]
    .map(pelicula_a_indice)
)

display(train_ranking.head())

,userId,movieId,label,usuario_idx,pelicula_idx
0,326,71462,1,1,1
1,326,5879,0,1,3530
2,326,1084,0,1,227
3,326,45950,0,1,2584
4,326,80241,0,1,3980


In [13]:
print(
    train_ranking[
        ["usuario_idx", "pelicula_idx"]
    ].isna().sum()
)

usuario_idx     0
pelicula_idx    0
dtype: int64


## 5. Construcción del modelo de ranking

El modelo recibe como entradas el usuario y la película y aprende una representación latente de ambos mediante embeddings.

A diferencia del modelo anterior de regresión, la salida representa una probabilidad de relevancia entre 0 y 1. El objetivo es asignar puntuaciones altas a las interacciones positivas y puntuaciones bajas a los ejemplos negativos generados mediante muestreo.

In [14]:
embedding_dim = 16

entrada_usuario = keras.Input(
    shape=(1,),
    name="usuario"
)

entrada_pelicula = keras.Input(
    shape=(1,),
    name="pelicula"
)

In [15]:
embedding_usuario = layers.Embedding(
    input_dim=num_usuarios,
    output_dim=embedding_dim,
    name="embedding_usuario"
)(entrada_usuario)

embedding_pelicula = layers.Embedding(
    input_dim=num_peliculas,
    output_dim=embedding_dim,
    name="embedding_pelicula"
)(entrada_pelicula)

vector_usuario = layers.Flatten()(embedding_usuario)
vector_pelicula = layers.Flatten()(embedding_pelicula)

In [16]:
x = layers.Concatenate()([
    vector_usuario,
    vector_pelicula
])

x = layers.Dense(
    64,
    activation="relu"
)(x)

x = layers.Dense(
    32,
    activation="relu"
)(x)

In [17]:
salida = layers.Dense(
    1,
    activation="sigmoid",
    name="relevancia"
)(x)

In [18]:
modelo_ranking = keras.Model(
    inputs=[
        entrada_usuario,
        entrada_pelicula
    ],
    outputs=salida
)

modelo_ranking.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ usuario             │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pelicula            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_usuario   │ (None, 1, 16)     │      9,760 │ usuario[0][0]     │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_pelicula  │ (None, 1, 16)     │    143,728 │ pelicula[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 16)        │          0 │ embedding_usuari… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 16)        │          0 │ embedding_pelicu… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 32)        │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      2,112 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │      2,080 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relevancia (Dense)  │ (None, 1)         │         33 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 157,713 (616.07 KB)

 Trainable params: 157,713 (616.07 KB)

 Non-trainable params: 0 (0.00 B)

### 5.1. Compilación del modelo

Al tratarse de un problema de clasificación binaria se utiliza la entropía cruzada binaria como función de pérdida.

Esta función penaliza al modelo cuando asigna baja probabilidad a las interacciones positivas o alta probabilidad a los ejemplos negativos.

In [19]:
modelo_ranking.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

## 6. Preparación de los datos de entrenamiento y validación

Los ejemplos generados se transforman en arrays que pueden ser utilizados directamente por la red neuronal.

Debido al desequilibrio introducido por el muestreo de cuatro ejemplos negativos por cada interacción positiva, la accuracy no se utilizará como criterio principal para determinar la calidad del recomendador. La evaluación final se realizará mediante métricas específicas de ranking como Hit Rate@10 y NDCG@10.

In [20]:
X_train_usuario = train_ranking[
    "usuario_idx"
].to_numpy().astype("int32")

X_train_pelicula = train_ranking[
    "pelicula_idx"
].to_numpy().astype("int32")

y_train = train_ranking[
    "label"
].to_numpy().astype("float32")

print("Usuarios train:", X_train_usuario.shape)
print("Películas train:", X_train_pelicula.shape)
print("Labels train:", y_train.shape)

Usuarios train: (194445,)
Películas train: (194445,)
Labels train: (194445,)


In [21]:
rng_val = np.random.default_rng(43)

ejemplos_val = []

for fila in validation_positivos.itertuples():

    usuario = fila.userId
    pelicula_positiva = fila.movieId

    # Solo podemos utilizar películas conocidas por el modelo
    if pelicula_positiva not in pelicula_a_indice:
        continue

    ejemplos_val.append(
        (usuario, pelicula_positiva, 1)
    )

    vistas = peliculas_vistas_usuario[usuario]

    negativas_disponibles = peliculas_train[
        ~np.isin(
            peliculas_train,
            list(vistas)
        )
    ]

    peliculas_negativas = rng_val.choice(
        negativas_disponibles,
        size=4,
        replace=False
    )

    for pelicula_negativa in peliculas_negativas:

        ejemplos_val.append(
            (usuario, pelicula_negativa, 0)
        )

In [22]:
validation_ranking = pd.DataFrame(
    ejemplos_val,
    columns=[
        "userId",
        "movieId",
        "label"
    ]
)

validation_ranking["usuario_idx"] = (
    validation_ranking["userId"]
    .map(usuario_a_indice)
)

validation_ranking["pelicula_idx"] = (
    validation_ranking["movieId"]
    .map(pelicula_a_indice)
)

print(
    "Tamaño validation ranking:",
    validation_ranking.shape
)

print()
print(
    validation_ranking["label"].value_counts()
)

Tamaño validation ranking: (23820, 5)

label
0    19056
1     4764
Name: count, dtype: int64


In [23]:
X_val_usuario = validation_ranking[
    "usuario_idx"
].to_numpy().astype("int32")

X_val_pelicula = validation_ranking[
    "pelicula_idx"
].to_numpy().astype("int32")

y_val = validation_ranking[
    "label"
].to_numpy().astype("float32")

print("Usuarios validation:", X_val_usuario.shape)
print("Películas validation:", X_val_pelicula.shape)
print("Labels validation:", y_val.shape)

Usuarios validation: (23820,)
Películas validation: (23820,)
Labels validation: (23820,)


## 7. Entrenamiento del modelo

El modelo se entrena utilizando los ejemplos positivos y negativos generados anteriormente.

Se utiliza Early Stopping sobre la pérdida de validación para detener el entrenamiento cuando el modelo deja de mejorar y restaurar automáticamente los pesos correspondientes a la mejor época.

In [24]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

In [25]:
history_ranking = modelo_ranking.fit(
    x=[
        X_train_usuario,
        X_train_pelicula
    ],
    y=y_train,
    validation_data=(
        [
            X_val_usuario,
            X_val_pelicula
        ],
        y_val
    ),
    epochs=30,
    batch_size=256,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/30
760/760 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.8538 - loss: 0.3638 - val_accuracy: 0.8752 - val_loss: 0.2949
Epoch 2/30
760/760 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8746 - loss: 0.2904 - val_accuracy: 0.8757 - val_loss: 0.2927
Epoch 3/30
760/760 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8758 - loss: 0.2808 - val_accuracy: 0.8780 - val_loss: 0.2965
Epoch 4/30
760/760 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8769 - loss: 0.2741 - val_accuracy: 0.8773 - val_loss: 0.3043
Epoch 5/30
760/760 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8787 - loss: 0.2654 - val_accuracy: 0.8770 - val_loss: 0.3225


## 8. Evaluación Top-N del modelo orientado a ranking

Para evaluar el nuevo modelo se reproduce el mismo protocolo utilizado en la comparación de los modelos anteriores.

Para cada interacción relevante del conjunto de test se construye un conjunto formado por la película relevante y 99 películas no valoradas por el usuario. De esta forma, los resultados obtenidos pueden compararse directamente con los modelos de filtrado colaborativo, factores latentes y Deep Learning híbrido.

In [26]:
usuarios_train = set(train["userId"])
peliculas_train_set = set(train["movieId"])

test_comun = test[
    test["userId"].isin(usuarios_train)
    & test["movieId"].isin(peliculas_train_set)
].copy()

print("Test común:", len(test_comun))

Test común: 9695


In [27]:
umbral_relevancia = 4.0

test_relevantes = test_comun[
    test_comun["rating"] >= umbral_relevancia
].copy()

print(
    "Interacciones relevantes:",
    len(test_relevantes)
)

Interacciones relevantes: 4617


In [28]:
valoraciones_conocidas = pd.concat(
    [
        train[["userId", "movieId"]],
        validation[["userId", "movieId"]],
        test[["userId", "movieId"]]
    ],
    ignore_index=True
)

peliculas_vistas_usuario = (
    valoraciones_conocidas
    .groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

In [29]:
peliculas_candidatas = np.array(
    sorted(peliculas_train_set)
)

rng = np.random.default_rng(42)

candidatos_topn = []

for fila in test_relevantes.itertuples():

    usuario = fila.userId
    pelicula_relevante = fila.movieId

    vistas = peliculas_vistas_usuario.get(
        usuario,
        set()
    )

    negativas_disponibles = np.array([
        pelicula
        for pelicula in peliculas_candidatas
        if pelicula not in vistas
    ])

    negativas = rng.choice(
        negativas_disponibles,
        size=99,
        replace=False
    )

    candidatos = np.concatenate(
        [
            [pelicula_relevante],
            negativas
        ]
    )

    candidatos_topn.append({
        "userId": usuario,
        "pelicula_relevante": pelicula_relevante,
        "candidatos": candidatos
    })

In [30]:
print(
    "Casos de evaluación:",
    len(candidatos_topn)
)

print(
    "Candidatos del primer caso:",
    len(candidatos_topn[0]["candidatos"])
)

print(
    "Usuario del primer caso:",
    candidatos_topn[0]["userId"]
)

print(
    "Película relevante:",
    candidatos_topn[0]["pelicula_relevante"]
)

Casos de evaluación: 4617
Candidatos del primer caso: 100
Usuario del primer caso: 491
Película relevante: 30825


In [31]:
usuarios_topn = []
peliculas_topn = []

for caso in candidatos_topn:

    usuario = caso["userId"]

    for pelicula in caso["candidatos"]:

        usuarios_topn.append(
            usuario_a_indice[usuario]
        )

        peliculas_topn.append(
            pelicula_a_indice[pelicula]
        )

usuarios_topn = np.array(
    usuarios_topn,
    dtype="int32"
)

peliculas_topn = np.array(
    peliculas_topn,
    dtype="int32"
)

print("Usuarios:", usuarios_topn.shape)
print("Películas:", peliculas_topn.shape)

Usuarios: (461700,)
Películas: (461700,)


In [32]:
predicciones_ranking = modelo_ranking.predict(
    [
        usuarios_topn,
        peliculas_topn
    ],
    batch_size=1024,
    verbose=1
).flatten()

print(
    "Número de predicciones:",
    len(predicciones_ranking)
)

print(
    "Predicción mínima:",
    round(predicciones_ranking.min(), 4)
)

print(
    "Predicción máxima:",
    round(predicciones_ranking.max(), 4)
)

451/451 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
Número de predicciones: 461700
Predicción mínima: 1e-04
Predicción máxima: 0.9926


In [33]:
predicciones_ranking = predicciones_ranking.reshape(
    len(candidatos_topn),
    100
)

rangos_ranking = []
hits_ranking = []

for idx, caso in enumerate(candidatos_topn):

    candidatos = caso["candidatos"]
    pelicula_relevante = caso["pelicula_relevante"]

    puntuaciones = predicciones_ranking[idx]

    orden = np.argsort(-puntuaciones)
    ranking = candidatos[orden]

    posicion = (
        np.where(
            ranking == pelicula_relevante
        )[0][0]
        + 1
    )

    rangos_ranking.append(posicion)

    hits_ranking.append(
        1 if posicion <= 10 else 0
    )

In [34]:
hit_rate_ranking = np.mean(
    hits_ranking
)

ndcg_ranking = np.mean([
    1 / np.log2(posicion + 1)
    if posicion <= 10
    else 0
    for posicion in rangos_ranking
])

print(
    "Hit Rate@10 DL Ranking:",
    round(hit_rate_ranking, 4)
)

print(
    "NDCG@10 DL Ranking:",
    round(ndcg_ranking, 4)
)

print(
    "Posición media:",
    round(np.mean(rangos_ranking), 2)
)

print(
    "Posición mediana:",
    round(np.median(rangos_ranking), 2)
)

Hit Rate@10 DL Ranking: 0.7158
NDCG@10 DL Ranking: 0.4681
Posición media: 11.07
Posición mediana: 4.0


## 9. Validación del protocolo de evaluación

Dado el elevado rendimiento obtenido por el modelo orientado a ranking, se realizan comprobaciones adicionales sobre el protocolo experimental.

El objetivo es verificar que los conjuntos de candidatos se han construido correctamente y descartar posibles fugas de información o situaciones que puedan favorecer artificialmente al modelo.

### 9.1. Comprobación de los candidatos negativos

Se comprueba que las 99 películas utilizadas como ejemplos negativos en cada caso de evaluación no hayan sido valoradas previamente por el usuario en ninguno de los conjuntos de datos.

In [35]:
errores_negativos = 0
errores_relevante = 0

for caso in candidatos_topn:

    usuario = caso["userId"]
    relevante = caso["pelicula_relevante"]
    candidatos = caso["candidatos"]

    # La primera película debería ser la relevante
    if candidatos[0] != relevante:
        errores_relevante += 1

    vistas = peliculas_vistas_usuario[usuario]

    # Excluimos la relevante
    negativos = candidatos[
        candidatos != relevante
    ]

    # Ningún negativo debería estar entre las vistas
    for pelicula in negativos:

        if pelicula in vistas:
            errores_negativos += 1


print(
    "Casos con relevante incorrecta:",
    errores_relevante
)

print(
    "Negativos que realmente estaban vistos:",
    errores_negativos
)

Casos con relevante incorrecta: 0
Negativos que realmente estaban vistos: 0


In [36]:
casos_con_duplicados = 0

for caso in candidatos_topn:

    candidatos = caso["candidatos"]

    if len(np.unique(candidatos)) != 100:
        casos_con_duplicados += 1


print(
    "Casos con candidatos duplicados:",
    casos_con_duplicados
)

Casos con candidatos duplicados: 0


In [37]:
positivos_train_set = set(
    zip(
        train_ranking.loc[
            train_ranking["label"] == 1,
            "userId"
        ],
        train_ranking.loc[
            train_ranking["label"] == 1,
            "movieId"
        ]
    )
)

fugas_test_train = 0

for fila in test_relevantes.itertuples():

    if (
        fila.userId,
        fila.movieId
    ) in positivos_train_set:

        fugas_test_train += 1


print(
    "Interacciones relevantes de test presentes "
    "como positivas en entrenamiento:",
    fugas_test_train
)

Interacciones relevantes de test presentes como positivas en entrenamiento: 0


### 9.4. Comparación con un baseline de popularidad

Para comprobar si el rendimiento obtenido por el modelo puede explicarse simplemente por la popularidad de las películas, se construye un baseline que ordena los candidatos únicamente según el número de valoraciones recibidas en el conjunto de entrenamiento.

Este modelo no utiliza información específica del usuario. Por tanto, su comparación con el modelo neuronal permite analizar si la personalización aprendida aporta una mejora respecto a una estrategia basada exclusivamente en popularidad.

In [38]:
popularidad = (
    train
    .groupby("movieId")
    .size()
    .to_dict()
)

In [39]:
rangos_popularidad = []
hits_popularidad = []

for caso in candidatos_topn:

    candidatos = caso["candidatos"]
    pelicula_relevante = caso["pelicula_relevante"]

    puntuaciones = np.array([
        popularidad.get(pelicula, 0)
        for pelicula in candidatos
    ])

    orden = np.argsort(-puntuaciones)

    ranking = candidatos[orden]

    posicion = (
        np.where(
            ranking == pelicula_relevante
        )[0][0]
        + 1
    )

    rangos_popularidad.append(posicion)

    hits_popularidad.append(
        1 if posicion <= 10 else 0
    )

In [40]:
hit_rate_popularidad = np.mean(
    hits_popularidad
)

ndcg_popularidad = np.mean([
    1 / np.log2(posicion + 1)
    if posicion <= 10
    else 0
    for posicion in rangos_popularidad
])

print(
    "Hit Rate@10 Popularidad:",
    round(hit_rate_popularidad, 4)
)

print(
    "NDCG@10 Popularidad:",
    round(ndcg_popularidad, 4)
)

print(
    "Posición media:",
    round(np.mean(rangos_popularidad), 2)
)

print(
    "Posición mediana:",
    round(np.median(rangos_popularidad), 2)
)

Hit Rate@10 Popularidad: 0.7102
NDCG@10 Popularidad: 0.4724
Posición media: 10.82
Posición mediana: 4.0


In [41]:
conteo_valoraciones = (
    train
    .groupby("movieId")
    .size()
)

print(
    "Películas totales:",
    len(conteo_valoraciones)
)

for minimo in [5, 10, 20, 30, 50]:

    print(
        f"Con al menos {minimo} valoraciones:",
        (conteo_valoraciones >= minimo).sum()
    )

Películas totales: 8983
Con al menos 5 valoraciones: 3219
Con al menos 10 valoraciones: 1921
Con al menos 20 valoraciones: 1053
Con al menos 30 valoraciones: 697
Con al menos 50 valoraciones: 325


### 9.5. Evaluación con negativos de mayor popularidad

La evaluación inicial mostró que un baseline basado únicamente en popularidad obtenía resultados muy próximos al modelo Deep Learning orientado a ranking.

Esto indica que el muestreo completamente aleatorio de películas negativas genera numerosos candidatos poco populares, facilitando la identificación de la película relevante.

Para realizar una evaluación más exigente, se construye un segundo escenario en el que las películas negativas deben disponer de al menos 20 valoraciones en el conjunto de entrenamiento. De esta forma, el modelo debe distinguir la película relevante frente a alternativas con una presencia significativa en el conjunto de datos.

In [42]:
peliculas_populares = np.array(
    conteo_valoraciones[
        conteo_valoraciones >= 20
    ].index
)

print(
    "Películas candidatas populares:",
    len(peliculas_populares)
)

Películas candidatas populares: 1053


In [43]:
rng_dificil = np.random.default_rng(42)

candidatos_topn_dificil = []
casos_descartados = 0

for fila in test_relevantes.itertuples():

    usuario = fila.userId
    pelicula_relevante = fila.movieId

    vistas = peliculas_vistas_usuario.get(
        usuario,
        set()
    )

    negativas_disponibles = np.array([
        pelicula
        for pelicula in peliculas_populares
        if pelicula not in vistas
        and pelicula != pelicula_relevante
    ])

    if len(negativas_disponibles) < 99:
        casos_descartados += 1
        continue

    negativas = rng_dificil.choice(
        negativas_disponibles,
        size=99,
        replace=False
    )

    candidatos = np.concatenate([
        [pelicula_relevante],
        negativas
    ])

    candidatos_topn_dificil.append({
        "userId": usuario,
        "pelicula_relevante": pelicula_relevante,
        "candidatos": candidatos
    })


print(
    "Casos de evaluación:",
    len(candidatos_topn_dificil)
)

print(
    "Casos descartados:",
    casos_descartados
)

Casos de evaluación: 4617
Casos descartados: 0


In [44]:
rangos_pop_dificil = []
hits_pop_dificil = []

for caso in candidatos_topn_dificil:

    candidatos = caso["candidatos"]
    pelicula_relevante = caso["pelicula_relevante"]

    puntuaciones = np.array([
        popularidad.get(pelicula, 0)
        for pelicula in candidatos
    ])

    orden = np.argsort(-puntuaciones)
    ranking = candidatos[orden]

    posicion = (
        np.where(
            ranking == pelicula_relevante
        )[0][0]
        + 1
    )

    rangos_pop_dificil.append(posicion)

    hits_pop_dificil.append(
        1 if posicion <= 10 else 0
    )


hr_pop_dificil = np.mean(hits_pop_dificil)

ndcg_pop_dificil = np.mean([
    1 / np.log2(posicion + 1)
    if posicion <= 10
    else 0
    for posicion in rangos_pop_dificil
])

print(
    "HR@10 Popularidad difícil:",
    round(hr_pop_dificil, 4)
)

print(
    "NDCG@10 Popularidad difícil:",
    round(ndcg_pop_dificil, 4)
)

print(
    "Posición media:",
    round(np.mean(rangos_pop_dificil), 2)
)

print(
    "Posición mediana:",
    round(np.median(rangos_pop_dificil), 2)
)

HR@10 Popularidad difícil: 0.2811
NDCG@10 Popularidad difícil: 0.1656
Posición media: 48.27
Posición mediana: 37.0


In [45]:
usuarios_dificil = []
peliculas_dificil = []

for caso in candidatos_topn_dificil:

    usuario = caso["userId"]

    for pelicula in caso["candidatos"]:

        usuarios_dificil.append(
            usuario_a_indice[usuario]
        )

        peliculas_dificil.append(
            pelicula_a_indice[pelicula]
        )

usuarios_dificil = np.array(
    usuarios_dificil,
    dtype="int32"
)

peliculas_dificil = np.array(
    peliculas_dificil,
    dtype="int32"
)

print("Usuarios:", usuarios_dificil.shape)
print("Películas:", peliculas_dificil.shape)

Usuarios: (461700,)
Películas: (461700,)


In [46]:
pred_dl_dificil = modelo_ranking.predict(
    [
        usuarios_dificil,
        peliculas_dificil
    ],
    batch_size=1024,
    verbose=1
).flatten()

pred_dl_dificil = pred_dl_dificil.reshape(
    len(candidatos_topn_dificil),
    100
)

451/451 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [47]:
rangos_dl_dificil = []
hits_dl_dificil = []

for idx, caso in enumerate(
    candidatos_topn_dificil
):

    candidatos = caso["candidatos"]
    pelicula_relevante = caso["pelicula_relevante"]

    puntuaciones = pred_dl_dificil[idx]

    orden = np.argsort(-puntuaciones)
    ranking = candidatos[orden]

    posicion = (
        np.where(
            ranking == pelicula_relevante
        )[0][0]
        + 1
    )

    rangos_dl_dificil.append(posicion)

    hits_dl_dificil.append(
        1 if posicion <= 10 else 0
    )

In [48]:
hr_dl_dificil = np.mean(
    hits_dl_dificil
)

ndcg_dl_dificil = np.mean([
    1 / np.log2(posicion + 1)
    if posicion <= 10
    else 0
    for posicion in rangos_dl_dificil
])

print(
    "HR@10 DL Ranking difícil:",
    round(hr_dl_dificil, 4)
)

print(
    "NDCG@10 DL Ranking difícil:",
    round(ndcg_dl_dificil, 4)
)

print(
    "Posición media:",
    round(np.mean(rangos_dl_dificil), 2)
)

print(
    "Posición mediana:",
    round(np.median(rangos_dl_dificil), 2)
)

HR@10 DL Ranking difícil: 0.2777
NDCG@10 DL Ranking difícil: 0.1646
Posición media: 45.81
Posición mediana: 37.0


## 10. Entrenamiento con negativos difíciles

Los resultados anteriores mostraron que el modelo orientado a ranking obtenía un rendimiento muy similar a un baseline basado exclusivamente en popularidad.

Una posible causa es el uso de películas negativas seleccionadas aleatoriamente durante el entrenamiento. Este procedimiento puede facilitar que el modelo diferencie las interacciones positivas utilizando principalmente la popularidad de las películas.

Para reducir este efecto se desarrolla una segunda variante en la que los ejemplos negativos se seleccionan entre películas con al menos 20 valoraciones en el conjunto de entrenamiento. De esta forma, el modelo debe distinguir las películas relevantes frente a alternativas con un nivel mínimo de popularidad.

In [49]:
rng_train_dificil = np.random.default_rng(42)

ejemplos_dificiles = []

for fila in train_positivos.itertuples():

    usuario = fila.userId
    pelicula_positiva = fila.movieId

    # Ejemplo positivo
    ejemplos_dificiles.append(
        (usuario, pelicula_positiva, 1)
    )

    vistas = peliculas_vistas_usuario[usuario]

    negativas_disponibles = np.array([
        pelicula
        for pelicula in peliculas_populares
        if pelicula not in vistas
        and pelicula != pelicula_positiva
    ])

    # Solo continuamos si existen suficientes negativas
    if len(negativas_disponibles) >= 4:

        negativas = rng_train_dificil.choice(
            negativas_disponibles,
            size=4,
            replace=False
        )

        for pelicula_negativa in negativas:

            ejemplos_dificiles.append(
                (usuario, pelicula_negativa, 0)
            )

In [50]:
rng_train_dificil = np.random.default_rng(42)

ejemplos_dificiles = []

for fila in train_positivos.itertuples():

    usuario = fila.userId
    pelicula_positiva = fila.movieId

    vistas = peliculas_vistas_usuario[usuario]

    negativas_disponibles = np.array([
        pelicula
        for pelicula in peliculas_populares
        if pelicula not in vistas
        and pelicula != pelicula_positiva
    ])

    if len(negativas_disponibles) < 4:
        continue

    # Positivo
    ejemplos_dificiles.append(
        (usuario, pelicula_positiva, 1)
    )

    # Cuatro negativos difíciles
    negativas = rng_train_dificil.choice(
        negativas_disponibles,
        size=4,
        replace=False
    )

    for pelicula_negativa in negativas:

        ejemplos_dificiles.append(
            (usuario, pelicula_negativa, 0)
        )

In [51]:
train_ranking_dificil = pd.DataFrame(
    ejemplos_dificiles,
    columns=[
        "userId",
        "movieId",
        "label"
    ]
)

print(
    "Tamaño:",
    train_ranking_dificil.shape
)

print()
print(
    train_ranking_dificil["label"].value_counts()
)

Tamaño: (194445, 3)

label
0    155556
1     38889
Name: count, dtype: int64


In [52]:
train_ranking_dificil["usuario_idx"] = (
    train_ranking_dificil["userId"]
    .map(usuario_a_indice)
)

train_ranking_dificil["pelicula_idx"] = (
    train_ranking_dificil["movieId"]
    .map(pelicula_a_indice)
)

X_train_usuario_dificil = (
    train_ranking_dificil["usuario_idx"]
    .to_numpy()
    .astype("int32")
)

X_train_pelicula_dificil = (
    train_ranking_dificil["pelicula_idx"]
    .to_numpy()
    .astype("int32")
)

y_train_dificil = (
    train_ranking_dificil["label"]
    .to_numpy()
    .astype("float32")
)

print("Usuarios:", X_train_usuario_dificil.shape)
print("Películas:", X_train_pelicula_dificil.shape)
print("Labels:", y_train_dificil.shape)

Usuarios: (194445,)
Películas: (194445,)
Labels: (194445,)


### 10.3. Construcción del modelo

Para comparar ambas estrategias de muestreo de forma controlada, se construye un nuevo modelo con la misma arquitectura utilizada anteriormente.

El modelo se inicializa desde cero, de forma que la única diferencia entre ambos experimentos sea el procedimiento utilizado para seleccionar los ejemplos negativos.

In [53]:
embedding_dim = 16

entrada_usuario_d = keras.Input(
    shape=(1,),
    name="usuario"
)

entrada_pelicula_d = keras.Input(
    shape=(1,),
    name="pelicula"
)

embedding_usuario_d = layers.Embedding(
    input_dim=num_usuarios,
    output_dim=embedding_dim,
    name="embedding_usuario"
)(entrada_usuario_d)

embedding_pelicula_d = layers.Embedding(
    input_dim=num_peliculas,
    output_dim=embedding_dim,
    name="embedding_pelicula"
)(entrada_pelicula_d)

vector_usuario_d = layers.Flatten()(
    embedding_usuario_d
)

vector_pelicula_d = layers.Flatten()(
    embedding_pelicula_d
)

x = layers.Concatenate()([
    vector_usuario_d,
    vector_pelicula_d
])

x = layers.Dense(
    64,
    activation="relu"
)(x)

x = layers.Dense(
    32,
    activation="relu"
)(x)

salida_d = layers.Dense(
    1,
    activation="sigmoid",
    name="relevancia"
)(x)

modelo_ranking_dificil = keras.Model(
    inputs=[
        entrada_usuario_d,
        entrada_pelicula_d
    ],
    outputs=salida_d
)

modelo_ranking_dificil.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

modelo_ranking_dificil.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ usuario             │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pelicula            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_usuario   │ (None, 1, 16)     │      9,760 │ usuario[0][0]     │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_pelicula  │ (None, 1, 16)     │    143,728 │ pelicula[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 16)        │          0 │ embedding_usuari… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 16)        │          0 │ embedding_pelicu… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 32)        │          0 │ flatten_2[0][0],  │
│ (Concatenate)       │                   │            │ flatten_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      2,112 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 32)        │      2,080 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relevancia (Dense)  │ (None, 1)         │         33 │ dense_3[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 157,713 (616.07 KB)

 Trainable params: 157,713 (616.07 KB)

 Non-trainable params: 0 (0.00 B)

In [54]:
rng_val_dificil = np.random.default_rng(43)

ejemplos_val_dificiles = []

for fila in validation_positivos.itertuples():

    usuario = fila.userId
    pelicula_positiva = fila.movieId

    if pelicula_positiva not in pelicula_a_indice:
        continue

    vistas = peliculas_vistas_usuario[usuario]

    negativas_disponibles = np.array([
        pelicula
        for pelicula in peliculas_populares
        if pelicula not in vistas
        and pelicula != pelicula_positiva
    ])

    if len(negativas_disponibles) < 4:
        continue

    ejemplos_val_dificiles.append(
        (usuario, pelicula_positiva, 1)
    )

    negativas = rng_val_dificil.choice(
        negativas_disponibles,
        size=4,
        replace=False
    )

    for pelicula_negativa in negativas:
        ejemplos_val_dificiles.append(
            (usuario, pelicula_negativa, 0)
        )

In [55]:
validation_ranking_dificil = pd.DataFrame(
    ejemplos_val_dificiles,
    columns=[
        "userId",
        "movieId",
        "label"
    ]
)

validation_ranking_dificil["usuario_idx"] = (
    validation_ranking_dificil["userId"]
    .map(usuario_a_indice)
)

validation_ranking_dificil["pelicula_idx"] = (
    validation_ranking_dificil["movieId"]
    .map(pelicula_a_indice)
)

X_val_usuario_dificil = (
    validation_ranking_dificil["usuario_idx"]
    .to_numpy()
    .astype("int32")
)

X_val_pelicula_dificil = (
    validation_ranking_dificil["pelicula_idx"]
    .to_numpy()
    .astype("int32")
)

y_val_dificil = (
    validation_ranking_dificil["label"]
    .to_numpy()
    .astype("float32")
)

print(
    "Validation difícil:",
    validation_ranking_dificil.shape
)

print(
    validation_ranking_dificil["label"]
    .value_counts()
)

Validation difícil: (23820, 5)
label
0    19056
1     4764
Name: count, dtype: int64


### 10.5. Entrenamiento del modelo

El nuevo modelo se entrena utilizando ejemplos negativos de mayor dificultad tanto en entrenamiento como en validación.

Se mantiene la misma configuración de entrenamiento que en el modelo anterior para que la comparación se centre exclusivamente en el efecto producido por la estrategia de muestreo negativo.

In [56]:
early_stopping_dificil = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

In [57]:
history_ranking_dificil = modelo_ranking_dificil.fit(
    x=[
        X_train_usuario_dificil,
        X_train_pelicula_dificil
    ],
    y=y_train_dificil,
    validation_data=(
        [
            X_val_usuario_dificil,
            X_val_pelicula_dificil
        ],
        y_val_dificil
    ),
    epochs=30,
    batch_size=256,
    callbacks=[early_stopping_dificil],
    verbose=1
)

Epoch 1/30
760/760 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.8422 - loss: 0.3983 - val_accuracy: 0.8670 - val_loss: 0.3452
Epoch 2/30
760/760 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8720 - loss: 0.3374 - val_accuracy: 0.8735 - val_loss: 0.3353
Epoch 3/30
760/760 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8758 - loss: 0.3274 - val_accuracy: 0.8741 - val_loss: 0.3311
Epoch 4/30
760/760 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8789 - loss: 0.3183 - val_accuracy: 0.8743 - val_loss: 0.3224
Epoch 5/30
760/760 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8832 - loss: 0.2968 - val_accuracy: 0.8793 - val_loss: 0.3030
Epoch 6/30
760/760 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8888 - loss: 0.2771 - val_accuracy: 0.8809 - val_loss: 0.2916
Epoch 7/30
760/760 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8932 - loss: 0.2627 - val_accuracy: 0.8843 - val_loss: 0.2835
Epoch 8/30
760/760 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8977 - loss: 0.2506 - val_accuracy: 0.

In [58]:
pred_dl_entrenado_dificil = modelo_ranking_dificil.predict(
    [
        usuarios_dificil,
        peliculas_dificil
    ],
    batch_size=1024,
    verbose=1
).flatten()

pred_dl_entrenado_dificil = (
    pred_dl_entrenado_dificil.reshape(
        len(candidatos_topn_dificil),
        100
    )
)

451/451 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [59]:
rangos_dl_entrenado_dificil = []
hits_dl_entrenado_dificil = []

for idx, caso in enumerate(
    candidatos_topn_dificil
):

    candidatos = caso["candidatos"]
    pelicula_relevante = caso["pelicula_relevante"]

    puntuaciones = (
        pred_dl_entrenado_dificil[idx]
    )

    orden = np.argsort(-puntuaciones)

    ranking = candidatos[orden]

    posicion = (
        np.where(
            ranking == pelicula_relevante
        )[0][0]
        + 1
    )

    rangos_dl_entrenado_dificil.append(
        posicion
    )

    hits_dl_entrenado_dificil.append(
        1 if posicion <= 10 else 0
    )

In [60]:
hr_dl_entrenado_dificil = np.mean(
    hits_dl_entrenado_dificil
)

ndcg_dl_entrenado_dificil = np.mean([
    1 / np.log2(posicion + 1)
    if posicion <= 10
    else 0
    for posicion
    in rangos_dl_entrenado_dificil
])

print(
    "HR@10 DL entrenado con negativos difíciles:",
    round(hr_dl_entrenado_dificil, 4)
)

print(
    "NDCG@10 DL entrenado con negativos difíciles:",
    round(ndcg_dl_entrenado_dificil, 4)
)

print(
    "Posición media:",
    round(
        np.mean(rangos_dl_entrenado_dificil),
        2
    )
)

print(
    "Posición mediana:",
    round(
        np.median(rangos_dl_entrenado_dificil),
        2
    )
)

HR@10 DL entrenado con negativos difíciles: 0.7219
NDCG@10 DL entrenado con negativos difíciles: 0.5612
Posición media: 10.52
Posición mediana: 2.0


## 11. Corrección de posibles fugas de información

Durante la generación inicial de ejemplos negativos se utilizó el conjunto completo de películas valoradas por cada usuario, incluyendo información procedente de entrenamiento, validación y test.

Aunque las interacciones de test no fueron utilizadas como ejemplos positivos durante el entrenamiento, esta información permitía conocer qué películas había valorado posteriormente cada usuario y evitar seleccionarlas como negativas.

Para garantizar un aislamiento estricto del conjunto de test, se reconstruye el muestreo negativo utilizando únicamente la información disponible en cada fase del proceso.

In [61]:
vistas_train_usuario = (
    train
    .groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

In [62]:
datos_hasta_validation = pd.concat(
    [
        train[["userId", "movieId"]],
        validation[["userId", "movieId"]]
    ],
    ignore_index=True
)

vistas_validation_usuario = (
    datos_hasta_validation
    .groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

In [63]:
print(
    "Usuarios con historial train:",
    len(vistas_train_usuario)
)

print(
    "Usuarios hasta validation:",
    len(vistas_validation_usuario)
)

Usuarios con historial train: 610
Usuarios hasta validation: 610


In [64]:
rng_train_limpio = np.random.default_rng(42)

ejemplos_train_limpios = []

for fila in train_positivos.itertuples():

    usuario = fila.userId
    pelicula_positiva = fila.movieId

    vistas = vistas_train_usuario.get(
        usuario,
        set()
    )

    negativas_disponibles = np.array([
        pelicula
        for pelicula in peliculas_populares
        if pelicula not in vistas
        and pelicula != pelicula_positiva
    ])

    if len(negativas_disponibles) < 4:
        continue

    ejemplos_train_limpios.append(
        (usuario, pelicula_positiva, 1)
    )

    negativas = rng_train_limpio.choice(
        negativas_disponibles,
        size=4,
        replace=False
    )

    for pelicula in negativas:

        ejemplos_train_limpios.append(
            (usuario, pelicula, 0)
        )

In [65]:
train_ranking_limpio = pd.DataFrame(
    ejemplos_train_limpios,
    columns=[
        "userId",
        "movieId",
        "label"
    ]
)

print(
    "Train limpio:",
    train_ranking_limpio.shape
)

print(
    train_ranking_limpio["label"]
    .value_counts()
)

Train limpio: (194445, 3)
label
0    155556
1     38889
Name: count, dtype: int64


In [66]:
rng_val_limpio = np.random.default_rng(43)

ejemplos_val_limpios = []

for fila in validation_positivos.itertuples():

    usuario = fila.userId
    pelicula_positiva = fila.movieId

    if pelicula_positiva not in pelicula_a_indice:
        continue

    vistas = vistas_validation_usuario.get(
        usuario,
        set()
    )

    negativas_disponibles = np.array([
        pelicula
        for pelicula in peliculas_populares
        if pelicula not in vistas
        and pelicula != pelicula_positiva
    ])

    if len(negativas_disponibles) < 4:
        continue

    ejemplos_val_limpios.append(
        (usuario, pelicula_positiva, 1)
    )

    negativas = rng_val_limpio.choice(
        negativas_disponibles,
        size=4,
        replace=False
    )

    for pelicula in negativas:

        ejemplos_val_limpios.append(
            (usuario, pelicula, 0)
        )

In [67]:
validation_ranking_limpio = pd.DataFrame(
    ejemplos_val_limpios,
    columns=[
        "userId",
        "movieId",
        "label"
    ]
)

print(
    "Validation limpia:",
    validation_ranking_limpio.shape
)

print(
    validation_ranking_limpio["label"]
    .value_counts()
)

Validation limpia: (23820, 3)
label
0    19056
1     4764
Name: count, dtype: int64


In [68]:
# TRAIN

train_ranking_limpio["usuario_idx"] = (
    train_ranking_limpio["userId"]
    .map(usuario_a_indice)
)

train_ranking_limpio["pelicula_idx"] = (
    train_ranking_limpio["movieId"]
    .map(pelicula_a_indice)
)

X_train_usuario_limpio = (
    train_ranking_limpio["usuario_idx"]
    .to_numpy()
    .astype("int32")
)

X_train_pelicula_limpio = (
    train_ranking_limpio["pelicula_idx"]
    .to_numpy()
    .astype("int32")
)

y_train_limpio = (
    train_ranking_limpio["label"]
    .to_numpy()
    .astype("float32")
)

In [69]:
# VALIDATION

validation_ranking_limpio["usuario_idx"] = (
    validation_ranking_limpio["userId"]
    .map(usuario_a_indice)
)

validation_ranking_limpio["pelicula_idx"] = (
    validation_ranking_limpio["movieId"]
    .map(pelicula_a_indice)
)

X_val_usuario_limpio = (
    validation_ranking_limpio["usuario_idx"]
    .to_numpy()
    .astype("int32")
)

X_val_pelicula_limpio = (
    validation_ranking_limpio["pelicula_idx"]
    .to_numpy()
    .astype("int32")
)

y_val_limpio = (
    validation_ranking_limpio["label"]
    .to_numpy()
    .astype("float32")
)

In [70]:
print(
    X_train_usuario_limpio.shape,
    X_train_pelicula_limpio.shape,
    y_train_limpio.shape
)

print(
    X_val_usuario_limpio.shape,
    X_val_pelicula_limpio.shape,
    y_val_limpio.shape
)

(194445,) (194445,) (194445,)
(23820,) (23820,) (23820,)


In [71]:
embedding_dim = 16

entrada_usuario_l = keras.Input(
    shape=(1,),
    name="usuario"
)

entrada_pelicula_l = keras.Input(
    shape=(1,),
    name="pelicula"
)

embedding_usuario_l = layers.Embedding(
    input_dim=num_usuarios,
    output_dim=embedding_dim,
    name="embedding_usuario"
)(entrada_usuario_l)

embedding_pelicula_l = layers.Embedding(
    input_dim=num_peliculas,
    output_dim=embedding_dim,
    name="embedding_pelicula"
)(entrada_pelicula_l)

vector_usuario_l = layers.Flatten()(
    embedding_usuario_l
)

vector_pelicula_l = layers.Flatten()(
    embedding_pelicula_l
)

x = layers.Concatenate()([
    vector_usuario_l,
    vector_pelicula_l
])

x = layers.Dense(
    64,
    activation="relu"
)(x)

x = layers.Dense(
    32,
    activation="relu"
)(x)

salida_l = layers.Dense(
    1,
    activation="sigmoid",
    name="relevancia"
)(x)

modelo_ranking_limpio = keras.Model(
    inputs=[
        entrada_usuario_l,
        entrada_pelicula_l
    ],
    outputs=salida_l
)

modelo_ranking_limpio.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [72]:
early_stopping_limpio = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history_ranking_limpio = modelo_ranking_limpio.fit(
    [
        X_train_usuario_limpio,
        X_train_pelicula_limpio
    ],
    y_train_limpio,
    validation_data=(
        [
            X_val_usuario_limpio,
            X_val_pelicula_limpio
        ],
        y_val_limpio
    ),
    epochs=30,
    batch_size=256,
    callbacks=[early_stopping_limpio],
    verbose=1
)

Epoch 1/30
760/760 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.8405 - loss: 0.4023 - val_accuracy: 0.8647 - val_loss: 0.3502
Epoch 2/30
760/760 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8675 - loss: 0.3467 - val_accuracy: 0.8664 - val_loss: 0.3460
Epoch 3/30
760/760 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8702 - loss: 0.3415 - val_accuracy: 0.8644 - val_loss: 0.3472
Epoch 4/30
760/760 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8704 - loss: 0.3392 - val_accuracy: 0.8631 - val_loss: 0.3484
Epoch 5/30
760/760 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8712 - loss: 0.3364 - val_accuracy: 0.8634 - val_loss: 0.3468


In [73]:
pred_dl_limpio = modelo_ranking_limpio.predict(
    [
        usuarios_dificil,
        peliculas_dificil
    ],
    batch_size=1024,
    verbose=1
).flatten()

pred_dl_limpio = pred_dl_limpio.reshape(
    len(candidatos_topn_dificil),
    100
)

451/451 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [74]:
rangos_dl_limpio = []
hits_dl_limpio = []

for idx, caso in enumerate(candidatos_topn_dificil):

    candidatos = caso["candidatos"]
    pelicula_relevante = caso["pelicula_relevante"]

    puntuaciones = pred_dl_limpio[idx]

    orden = np.argsort(-puntuaciones)
    ranking = candidatos[orden]

    posicion = (
        np.where(
            ranking == pelicula_relevante
        )[0][0]
        + 1
    )

    rangos_dl_limpio.append(posicion)

    hits_dl_limpio.append(
        1 if posicion <= 10 else 0
    )

In [75]:
hr_dl_limpio = np.mean(
    hits_dl_limpio
)

ndcg_dl_limpio = np.mean([
    1 / np.log2(posicion + 1)
    if posicion <= 10
    else 0
    for posicion in rangos_dl_limpio
])

print(
    "HR@10 DL limpio:",
    round(hr_dl_limpio, 4)
)

print(
    "NDCG@10 DL limpio:",
    round(ndcg_dl_limpio, 4)
)

print(
    "Posición media:",
    round(np.mean(rangos_dl_limpio), 2)
)

print(
    "Posición mediana:",
    round(np.median(rangos_dl_limpio), 2)
)

HR@10 DL limpio: 0.5742
NDCG@10 DL limpio: 0.4504
Posición media: 18.93
Posición mediana: 6.0


## 12. Evaluación final con aislamiento del conjunto de test

Para realizar una evaluación estrictamente independiente, los candidatos negativos del conjunto de test se seleccionan utilizando únicamente el historial conocido antes de esta fase, compuesto por las interacciones de entrenamiento y validación.

De esta forma, ninguna interacción adicional del conjunto de test se utiliza para determinar qué películas pueden actuar como ejemplos negativos.

In [76]:
historial_pretest = pd.concat(
    [
        train[["userId", "movieId"]],
        validation[["userId", "movieId"]]
    ],
    ignore_index=True
)

vistas_pretest_usuario = (
    historial_pretest
    .groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

print(
    "Usuarios con historial pre-test:",
    len(vistas_pretest_usuario)
)

Usuarios con historial pre-test: 610


In [77]:
rng_test_limpio = np.random.default_rng(42)

candidatos_test_limpio = []
casos_descartados = 0

for fila in test_relevantes.itertuples():

    usuario = fila.userId
    pelicula_relevante = fila.movieId

    vistas_previas = vistas_pretest_usuario.get(
        usuario,
        set()
    )

    negativas_disponibles = np.array([
        pelicula
        for pelicula in peliculas_populares
        if pelicula not in vistas_previas
        and pelicula != pelicula_relevante
    ])

    if len(negativas_disponibles) < 99:
        casos_descartados += 1
        continue

    negativas = rng_test_limpio.choice(
        negativas_disponibles,
        size=99,
        replace=False
    )

    candidatos = np.concatenate([
        [pelicula_relevante],
        negativas
    ])

    candidatos_test_limpio.append({
        "userId": usuario,
        "pelicula_relevante": pelicula_relevante,
        "candidatos": candidatos
    })

print(
    "Casos de evaluación:",
    len(candidatos_test_limpio)
)

print(
    "Casos descartados:",
    casos_descartados
)

print(
    "Candidatos por caso:",
    len(candidatos_test_limpio[0]["candidatos"])
)

Casos de evaluación: 4617
Casos descartados: 0
Candidatos por caso: 100


In [78]:
errores_historial = 0
errores_relevante = 0
errores_duplicados = 0

for caso in candidatos_test_limpio:

    usuario = caso["userId"]
    relevante = caso["pelicula_relevante"]
    candidatos = caso["candidatos"]

    if candidatos[0] != relevante:
        errores_relevante += 1

    if len(np.unique(candidatos)) != 100:
        errores_duplicados += 1

    vistas_previas = vistas_pretest_usuario.get(
        usuario,
        set()
    )

    negativos = candidatos[
        candidatos != relevante
    ]

    for pelicula in negativos:
        if pelicula in vistas_previas:
            errores_historial += 1


print(
    "Negativos presentes en historial previo:",
    errores_historial
)

print(
    "Relevantes incorrectas:",
    errores_relevante
)

print(
    "Casos con duplicados:",
    errores_duplicados
)

Negativos presentes en historial previo: 0
Relevantes incorrectas: 0
Casos con duplicados: 0


In [79]:
rangos_pop_test_limpio = []
hits_pop_test_limpio = []

for caso in candidatos_test_limpio:

    candidatos = caso["candidatos"]
    pelicula_relevante = caso["pelicula_relevante"]

    puntuaciones = np.array([
        popularidad.get(pelicula, 0)
        for pelicula in candidatos
    ])

    orden = np.argsort(-puntuaciones)
    ranking = candidatos[orden]

    posicion = (
        np.where(
            ranking == pelicula_relevante
        )[0][0]
        + 1
    )

    rangos_pop_test_limpio.append(posicion)
    hits_pop_test_limpio.append(
        1 if posicion <= 10 else 0
    )

hr_pop_test_limpio = np.mean(hits_pop_test_limpio)

ndcg_pop_test_limpio = np.mean([
    1 / np.log2(posicion + 1)
    if posicion <= 10
    else 0
    for posicion in rangos_pop_test_limpio
])

print("HR@10 Popularidad test limpio:", round(hr_pop_test_limpio, 4))
print("NDCG@10 Popularidad test limpio:", round(ndcg_pop_test_limpio, 4))
print("Posición media:", round(np.mean(rangos_pop_test_limpio), 2))
print("Posición mediana:", round(np.median(rangos_pop_test_limpio), 2))

HR@10 Popularidad test limpio: 0.2712
NDCG@10 Popularidad test limpio: 0.1551
Posición media: 48.68
Posición mediana: 38.0


In [80]:
usuarios_test_limpio = []
peliculas_test_limpio = []

for caso in candidatos_test_limpio:

    usuario = caso["userId"]

    for pelicula in caso["candidatos"]:

        usuarios_test_limpio.append(
            usuario_a_indice[usuario]
        )

        peliculas_test_limpio.append(
            pelicula_a_indice[pelicula]
        )

usuarios_test_limpio = np.array(
    usuarios_test_limpio,
    dtype="int32"
)

peliculas_test_limpio = np.array(
    peliculas_test_limpio,
    dtype="int32"
)

print("Usuarios:", usuarios_test_limpio.shape)
print("Películas:", peliculas_test_limpio.shape)

Usuarios: (461700,)
Películas: (461700,)


In [81]:
pred_dl_test_limpio = modelo_ranking_limpio.predict(
    [
        usuarios_test_limpio,
        peliculas_test_limpio
    ],
    batch_size=1024,
    verbose=1
).flatten()

pred_dl_test_limpio = pred_dl_test_limpio.reshape(
    len(candidatos_test_limpio),
    100
)

451/451 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [82]:
rangos_dl_test_limpio = []
hits_dl_test_limpio = []

for idx, caso in enumerate(candidatos_test_limpio):

    candidatos = caso["candidatos"]
    pelicula_relevante = caso["pelicula_relevante"]

    puntuaciones = pred_dl_test_limpio[idx]

    orden = np.argsort(-puntuaciones)
    ranking = candidatos[orden]

    posicion = (
        np.where(
            ranking == pelicula_relevante
        )[0][0]
        + 1
    )

    rangos_dl_test_limpio.append(posicion)

    hits_dl_test_limpio.append(
        1 if posicion <= 10 else 0
    )

In [83]:
hr_dl_test_limpio = np.mean(
    hits_dl_test_limpio
)

ndcg_dl_test_limpio = np.mean([
    1 / np.log2(posicion + 1)
    if posicion <= 10
    else 0
    for posicion in rangos_dl_test_limpio
])

print(
    "HR@10 DL test limpio:",
    round(hr_dl_test_limpio, 4)
)

print(
    "NDCG@10 DL test limpio:",
    round(ndcg_dl_test_limpio, 4)
)

print(
    "Posición media:",
    round(np.mean(rangos_dl_test_limpio), 2)
)

print(
    "Posición mediana:",
    round(np.median(rangos_dl_test_limpio), 2)
)

HR@10 DL test limpio: 0.5612
NDCG@10 DL test limpio: 0.4388
Posición media: 19.24
Posición mediana: 7.0


In [84]:
rangos_pop_test_limpio = []
hits_pop_test_limpio = []

for caso in candidatos_test_limpio:

    candidatos = caso["candidatos"]
    pelicula_relevante = caso["pelicula_relevante"]

    puntuaciones = np.array([
        popularidad.get(pelicula, 0)
        for pelicula in candidatos
    ])

    orden = np.argsort(-puntuaciones)
    ranking = candidatos[orden]

    posicion = (
        np.where(
            ranking == pelicula_relevante
        )[0][0]
        + 1
    )

    rangos_pop_test_limpio.append(posicion)
    hits_pop_test_limpio.append(
        1 if posicion <= 10 else 0
    )

hr_pop_test_limpio = np.mean(
    hits_pop_test_limpio
)

ndcg_pop_test_limpio = np.mean([
    1 / np.log2(posicion + 1)
    if posicion <= 10
    else 0
    for posicion in rangos_pop_test_limpio
])

print(
    "HR@10 Popularidad test limpio:",
    round(hr_pop_test_limpio, 4)
)

print(
    "NDCG@10 Popularidad test limpio:",
    round(ndcg_pop_test_limpio, 4)
)

print(
    "Posición media:",
    round(np.mean(rangos_pop_test_limpio), 2)
)

print(
    "Posición mediana:",
    round(np.median(rangos_pop_test_limpio), 2)
)

HR@10 Popularidad test limpio: 0.2712
NDCG@10 Popularidad test limpio: 0.1551
Posición media: 48.68
Posición mediana: 38.0


In [85]:
modelo_ranking_limpio.save(
    f"{ruta_modelos}/modelo_deep_learning_ranking.keras"
)

In [86]:
import pickle

with open(
    f"{ruta_modelos}/usuario_a_indice_ranking.pkl",
    "wb"
) as f:
    pickle.dump(usuario_a_indice, f)

with open(
    f"{ruta_modelos}/pelicula_a_indice_ranking.pkl",
    "wb"
) as f:
    pickle.dump(pelicula_a_indice, f)

In [87]:
with open(
    f"{ruta_modelos}/popularidad.pkl",
    "wb"
) as f:
    pickle.dump(popularidad, f)

with open(
    f"{ruta_modelos}/peliculas_populares.pkl",
    "wb"
) as f:
    pickle.dump(peliculas_populares, f)

In [88]:
import os

for archivo in os.listdir(ruta_modelos):
    print(archivo)

modelo_deep_learning.keras
usuario_a_indice.pkl
pelicula_a_indice.pkl
columnas_generos.pkl
preferencias_usuario.csv
resultados_validacion.csv
train.csv
validation.csv
test.csv
modelo_deep_learning_ranking.keras
usuario_a_indice_ranking.pkl
pelicula_a_indice_ranking.pkl
popularidad.pkl
peliculas_populares.pkl


In [89]:
resultados_finales_dl = pd.DataFrame({
    "Modelo": [
        "Baseline popularidad",
        "Deep Learning ranking final"
    ],
    "HR@10": [
        0.2712,
        0.5612
    ],
    "NDCG@10": [
        0.1551,
        0.4388
    ],
    "Posicion_media": [
        48.68,
        19.24
    ],
    "Posicion_mediana": [
        38,
        7
    ]
})

resultados_finales_dl.to_csv(
    f"{ruta_modelos}/resultados_ranking_final.csv",
    index=False
)

display(resultados_finales_dl)

,Modelo,HR@10,NDCG@10,Posicion_media,Posicion_mediana
0,Baseline popularidad,0.2712,0.1551,48.68,38
1,Deep Learning ranking final,0.5612,0.4388,19.24,7
